In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.optimizers import Adam

# Check if dataset exist
dataset_path = r"C:\Users\HP\Downloads\seattle-weather.csv"

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving seattle-weather.csv to seattle-weather.csv


In [ ]:
data = pd.read_csv("seattle-weather.csv")

print(data.head())


         date  precipitation  temp_max  temp_min  wind  weather
0  2012-01-01            0.0      12.8       5.0   4.7  drizzle
1  2012-01-02           10.9      10.6       2.8   4.5     rain
2  2012-01-03            0.8      11.7       7.2   2.3     rain
3  2012-01-04           20.3      12.2       5.6   4.7     rain
4  2012-01-05            1.3       8.9       2.8   6.1     rain


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           1461 non-null   object 
 1   precipitation  1461 non-null   float64
 2   temp_max       1461 non-null   float64
 3   temp_min       1461 non-null   float64
 4   wind           1461 non-null   float64
 5   weather        1461 non-null   object 
dtypes: float64(4), object(2)
memory usage: 68.6+ KB


In [ ]:
data['date'] = pd.to_datetime(data['date'])
data.set_index('date', inplace=True)
data

,precipitation,temp_max,temp_min,wind,weather
date,,,,,
2012-01-01,0.0,12.8,5.0,4.7,drizzle
2012-01-02,10.9,10.6,2.8,4.5,rain
2012-01-03,0.8,11.7,7.2,2.3,rain
2012-01-04,20.3,12.2,5.6,4.7,rain
2012-01-05,1.3,8.9,2.8,6.1,rain
...,...,...,...,...,...
2015-12-27,8.6,4.4,1.7,2.9,rain
2015-12-28,1.5,5.0,1.7,1.3,rain
2015-12-29,0.0,7.2,0.6,2.6,fog


In [ ]:
#select the relvent fetures
features = ['precipitation', 'temp_max', 'temp_min', 'wind']
data =data[features]

data = data.fillna(method='ffill')


/tmp/ipykernel_2744/3778367351.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill')


In [ ]:
import numpy as np

#normalize

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

#create
X = []
Y = []

for i in range(30, len(data_scaled)):
    X.append(data_scaled[i-30:i])
    Y.append(data_scaled[i, 0])

X = np.array(X)
Y = np.array(Y)

In [ ]:
split=int(0.8*len(X))

X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

#Built RNN model
model = Sequential([
    SimpleRNN(64, activation='tanh', input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_3 (SimpleRNN)        │ (None, 64)             │         4,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,529 (25.50 KB)

 Trainable params: 6,529 (25.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X_train, Y_train, epochs = 60, batch_size = 16, validation_split = 0.1, verbose = 1)

y_pred = model.predict(X_test)

temp_max_scaler = MinMaxScaler()
temp_max_scaler.min_, temp_max_scaler.scale_ = scaler.min_[0], scaler.scale_[0]

y_pred_rescaled = y_pred / scaler.scale_[0]+ scaler.min_[0]
y_test_rescaled = Y_test / scaler.scale_[0]+ scaler.min_[0]

Epoch 1/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0180 - val_loss: 0.0242
Epoch 2/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0134 - val_loss: 0.0262
Epoch 3/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0121 - val_loss: 0.0242
Epoch 4/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0138 - val_loss: 0.0229
Epoch 5/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0116 - val_loss: 0.0211
Epoch 6/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0115 - val_loss: 0.0214
Epoch 7/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0110 - val_loss: 0.0214
Epoch 8/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0103 - val_loss: 0.0224
Epoch 9/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0106 - val_loss: 0.0218
Epoch 10/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0103 - val_loss: 0.0235
Epoch 11/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0100 - val_loss: 0.0222
Epoch 12/60
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0100 - v

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 

Model Performance:
MAE: 4.76
RMSE: 7.67
